In [1]:
# ======================================
# STEP 1: Install dependencies
# ======================================
!pip install openai python-dotenv nest_asyncio


# ======================================
# STEP 2: Load NVIDIA API
# ======================================
import os
from dotenv import load_dotenv
from openai import OpenAI
import asyncio
import nest_asyncio
import random

nest_asyncio.apply()

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)


# ======================================
# STEP 3: Mock Hospital Database
# ======================================
patients = {

    "P101": {

        "name": "Rahul",

        "appointment": "2 April 10:00 AM",

        "lab_report": "Glucose 180 mg/dL (High)",

        "medicine": "Metformin 500mg"
    }

}


# ======================================
# STEP 4: MCP Tools
# ======================================

# scheduling tool
async def book_appointment(payload):

    return f"📅 Appointment confirmed for {payload['patient']} on 5 April 11:00 AM"


# lab explanation tool
async def explain_lab_report(report):

    prompt = f"""
    Explain this medical report in simple language:

    {report}
    """

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[

            {"role":"user","content":prompt}
        ]

    )

    return response.choices[0].message.content


# medication reminder tool
async def medication_reminder(medicine):

    return f"💊 Reminder: take {medicine} every 8 hours"


# emergency triage tool
async def triage_case(symptoms):

    prompt = f"""
    classify severity:

    critical
    urgent
    routine

    symptoms:
    {symptoms}

    return only severity
    """

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[

            {"role":"user","content":prompt}
        ]

    )

    severity = response.choices[0].message.content.strip()

    return f"🚨 case classified as {severity}"


# followup automation
async def send_followup(patient):

    return f"📧 Follow-up message sent to {patient}"


# ======================================
# STEP 5: LLM decides which tool
# ======================================
def decide_action(query):

    prompt = f"""

    classify into ONE:

    appointment
    lab_report
    medication
    emergency
    followup

    query:
    {query}

    return only keyword
    """

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[

            {"role":"user","content":prompt}
        ]

    )

    return response.choices[0].message.content.lower()


# ======================================
# STEP 6: Agent workflow
# ======================================
async def healthcare_agent(query, patient_id):

    action = decide_action(query)

    patient = patients.get(patient_id)


    if not patient:

        return "patient not found"


    if "appointment" in action:

        return await book_appointment(

            {"patient": patient["name"]}
        )


    elif "lab" in action:

        return await explain_lab_report(

            patient["lab_report"]
        )


    elif "medication" in action:

        return await medication_reminder(

            patient["medicine"]
        )


    elif "emergency" in action:

        return await triage_case(query)


    elif "followup" in action:

        return await send_followup(

            patient["name"]
        )


    else:

        return "ask healthcare related query"


# ======================================
# STEP 7: Run Demo
# ======================================
async def run_demo():

    print("\n🏥 Healthcare AI Demo\n")


    queries = [

        "book appointment",

        "explain my lab report",

        "remind medicine dosage",

        "patient has chest pain emergency",

        "send followup instructions"
    ]


    for q in queries:

        result = await healthcare_agent(

            q,

            "P101"
        )

        print(f"\n🧑 Query: {q}")

        print(f"🤖 Response: {result}")


await run_demo()


🏥 Healthcare AI Demo


🧑 Query: book appointment
🤖 Response: 📅 Appointment confirmed for Rahul on 5 April 11:00 AM

🧑 Query: explain my lab report
🤖 Response: This medical report is showing the result of a blood test that measures the level of glucose (a type of sugar) in your blood.

The result is 180 mg/dL, which is higher than normal. A normal glucose level is usually below 140 mg/dL.

This means that you have high blood sugar. This could be a sign of diabetes or pre-diabetes, which is a condition where your body has trouble controlling blood sugar levels.

In simple terms, your body is not using insulin (a hormone that helps regulate blood sugar) properly, and as a result, your blood sugar levels are higher than they should be. This can lead to health problems if not managed properly.

It's important to talk to your doctor about this result and what it means for your health. They may recommend further testing or changes to your diet and lifestyle to help get your blood sugar level